### LangChain

In [84]:
# Run this once in the notebook to install the required packages.
# %pip install -U langchain langchain-openai langchain-text-splitters

from pathlib import Path
import httpx2
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_openai import ChatOpenAI
from rich import print
from pydantic import SecretStr
from langchain.agents import create_agent
from typing import List
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownTextSplitter, MarkdownHeaderTextSplitter

In [54]:
def print_conversation(messages:List[BaseMessage]) -> None:
    for message in messages:
       message.pretty_print() 


In [78]:
openai_api_key = SecretStr(Path('openai-secret-key-ai-integrations-developers.txt').read_text(encoding='utf-8').strip())
openai_model = ChatOpenAI(
    model_name='gpt-5-nano',
    openai_api_key=openai_api_key,
    reasoning_effort='low',
    http_client=httpx2.Client(trust_env=False),
)

### CAG (Cache Augmented Generation)

In [79]:
# Load the FAQ document as plain text
text_loader = TextLoader('FAQ.md', encoding='utf-8')
documents = text_loader.load()

In [80]:
type(documents)
print(documents[0].page_content)

# Често задавани въпроси: Vivacom телевизия

Кратък справочник по информация от [официалната страница с често задавани въпроси за телевизия на 
Vivacom](https://www.vivacom.bg/pomosht/chesto-zadavani-vuprosi/televiziya). Последна проверка: 7 септември 2026 г.

> Информацията е обобщена с учебна цел. За актуални цени, условия и техническа наличност винаги проверявайте 
официалния сайт на Vivacom.

## Какво е EON TV?

EON е телевизионна платформа на Vivacom за гледане на живи канали и съдържание по заявка на телевизор, мобилно 
устройство или компютър. Посочени са пакети EON LIGHT, EON FULL и EON PREMIUM. Услугата може да се предлага като 
интерактивна телевизия, сателитна телевизия (EON SAT) или предплатена Smart TV услуга.

## Каква е разликата между EON TV, EON SAT и предплатена EON TV?

- **EON TV** е интерактивна телевизия, която може да се комбинира с оптичен интернет от Vivacom или да се използва 
самостоятелно.
- **EON SAT** използва сателитен сигнал и може да предлага интерактивни функции при интернет връзка.
- **Предплатена EON TV** е безсрочна услуга без дългосрочен договор и без предоставено оборудване; използва се чрез
Smart TV приложение.

## Нужен ли е интернет от Vivacom?

Не непременно. EON TV може да се използва и с интернет от друг доставчик, стига да има активен абонамент и стабилна
интернет връзка. Възможните комбинации и технологията на свързване зависят от адреса.

## Как се заявява EON TV?

Услугите EON TV и EON SAT могат да се заявят през сайта на Vivacom, в магазин или чрез обслужване на клиенти. За 
предплатената услуга се избира план и брой Smart TV приложения онлайн, плаща се с карта при поръчка и след 
активацията има 30-дневен период за използване.

## На колко устройства може да се гледа?

Според FAQ страницата лимитите са:

- EON TV: до 5 телевизионни устройства;
- EON SAT: до 4 телевизионни устройства;
- предплатена EON TV: до 2 Smart TV приложения;
- EON TV с интернет от друг доставчик: до 2 телевизионни устройства;
- мобилното приложение може да се използва на до 3 избрани мобилни/компютърни устройства.

За Smart TV приложението трябва да се провери списъкът със съвместими модели.

## Какви функции предлага EON TV?

Сред описаните възможности са връщане на съдържание до 7 дни назад за избрани канали, видеотека, персонални 
профили, детски профил и избор между светъл и тъмен интерфейс. Наличността на конкретно съдържание и функции зависи
от плана и устройството.

## Как се управлява абонаментът?

За промяна на EON TV пакет, добавяне на услуги, смяна на собственост или адрес може да се използват каналите за 
обслужване на клиенти или магазин на Vivacom. Преминаване към пакет с по-малко канали и по-ниска месечна цена е 
възможно след изтичане на договора или през последните три месеца от него.

## Как се добавят допълнителни пакети и видеотека?

Допълнителни канали и филмови пакети могат да се заявяват през My Vivacom, в магазин или чрез обслужване на 
клиенти. Някои пакети важат само за интерактивна телевизия. Видеотеката предлага филми и сериали по абонамент или 
като единични заглавия; за част от съдържанието е възможно офлайн гледане.

## Как протича инсталацията?

Преди активация Vivacom проверява техническата възможност на адреса. Посещение може да се уговори през магазин, 
сайта или на 123. Компанията посочва, че изпраща напомняне преди посещението, а техникът инсталира необходимото 
оборудване, проверява свързаността и демонстрира услугата.

## Нужно ли е специално оборудване?

За договорните EON TV услуги Vivacom предоставя оборудване за срока на договора. Според FAQ не може да се използва 
друг приемник вместо предоставения от оператора. Свързването към телевизора може да бъде чрез подходящ наличен 
вход, например HDMI, SCART или композитна връзка; конкретното окабеляване се уточнява на място.

## Мога ли да използвам EON в чужбина?

Vivacom гарантира домашната телевизия на територията на България. На територията на Европейския съюз официалната 
страница посочва възможност за активиране

In [81]:
# `system_prompt` must be a single string. A list of strings becomes invalid
# OpenAI message content (it expects content-block objects when content is a list).
system_prompt = "You are a helpful assistant that can answer questions and perform tasks."
document_context = "\n\n".join(
    document.page_content.strip()
    for document in documents
    if document.page_content.strip()
)

if document_context:
    system_prompt += f"\n\nUse this reference material when it is relevant:\n{document_context}"

agent = create_agent(
    model=openai_model, 
    tools=[], 
    system_prompt=system_prompt,
    debug=True)
        

In [82]:
messages = [
HumanMessage(content="Какво е EON?'")
]

exploration_response = agent.invoke(input ={"messages": messages})

[values] {'messages': [HumanMessage(content="Какво е EON?'", additional_kwargs={}, response_metadata={}, id='93abafba-8308-4004-b560-f75218b0ee6f')]}
[updates] {'model': {'messages': [AIMessage(content='EON е телевизионна платформа на Vivacom за гледане на живи канали и съдържание по заявка (VOD) на телевизор, мобилно устройство или компютър. Налични са пакети EON LIGHT, EON FULL и EON PREMIUM и различни варианти на услугата:\n\n- EON TV: интерактивна телевизия (може да се използва с интернет от Vivacom или самостоятелно).\n- EON SAT: сателитна телевизия, която може да има интерактивни функции при интернет.\n- Предплатена EON TV: без дългосрочен договор и без предоставено оборудване; използва се чрез Smart TV приложение.\n\nХарактеристики и условия:\n- Не е задължително интернет от Vivacom; може да се използва и с интернет от друг доставчик.\n- Брой устройства за гледане: EON TV до 5 телевизора, EON SAT до 4 телевизора, предплатена EON TV до 2 Smart TV приложения (и други ограничени сл

In [83]:
print_conversation(exploration_response['messages'])

================================ Human Message =================================

Какво е EON?'
================================== Ai Message ==================================

EON е телевизионна платформа на Vivacom за гледане на живи канали и съдържание по заявка (VOD) на телевизор, мобилно устройство или компютър. Налични са пакети EON LIGHT, EON FULL и EON PREMIUM и различни варианти на услугата:

- EON TV: интерактивна телевизия (може да се използва с интернет от Vivacom или самостоятелно).
- EON SAT: сателитна телевизия, която може да има интерактивни функции при интернет.
- Предплатена EON TV: без дългосрочен договор и без предоставено оборудване; използва се чрез Smart TV приложение.

Характеристики и условия:
- Не е задължително интернет от Vivacom; може да се използва и с интернет от друг доставчик.
- Брой устройства за гледане: EON TV до 5 телевизора, EON SAT до 4 телевизора, предплатена EON TV до 2 Smart TV приложения (и други ограничени случаи).
- Мога да се ползват функц

### RAG with Markdown Spliiter

In [95]:
splitter = MarkdownHeaderTextSplitter(headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
], strip_headers=False)

chunked_documents = []
for d in documents: 
    chunked_documents.extend(splitter.split_text(d.page_content))

print(chunked_documents)

[
    Document(
        metadata={'Header 1': 'Често задавани въпроси: Vivacom телевизия'},
        page_content='# Често задавани въпроси: Vivacom телевизия  \nКратък справочник по информация от 
[официалната страница с често задавани въпроси за телевизия на 
Vivacom](https://www.vivacom.bg/pomosht/chesto-zadavani-vuprosi/televiziya). Последна проверка: 7 септември 2026 г.
\n> Информацията е обобщена с учебна цел. За актуални цени, условия и техническа наличност винаги проверявайте 
официалния сайт на Vivacom.'
    ),
    Document(
        metadata={'Header 1': 'Често задавани въпроси: Vivacom телевизия', 'Header 2': 'Какво е EON TV?'},
        page_content='## Какво е EON TV?  \nEON е телевизионна платформа на Vivacom за гледане на живи канали и 
съдържание по заявка на телевизор, мобилно устройство или компютър. Посочени са пакети EON LIGHT, EON FULL и EON 
PREMIUM. Услугата може да се предлага като интерактивна телевизия, сателитна телевизия (EON SAT) или предплатена 
Smart TV услуга.'
    ),
    Document(
        metadata={
            'Header 1': 'Често задавани въпроси: Vivacom телевизия',
            'Header 2': 'Каква е разликата между EON TV, EON SAT и предплатена EON TV?'
        },
        page_content='## Каква е разликата между EON TV, EON SAT и предплатена EON TV?  \n- **EON TV** е 
интерактивна телевизия, която може да се комбинира с оптичен интернет от Vivacom или да се използва 
самостоятелно.\n- **EON SAT** използва сателитен сигнал и може да предлага интерактивни функции при интернет 
връзка.\n- **Предплатена EON TV** е безсрочна услуга без дългосрочен договор и без предоставено оборудване; 
използва се чрез Smart TV приложение.'
    ),
    Document(
        metadata={
            'Header 1': 'Често задавани въпроси: Vivacom телевизия',
            'Header 2': 'Нужен ли е интернет от Vivacom?'
        },
        page_content='## Нужен ли е интернет от Vivacom?  \nНе непременно. EON TV може да се използва и с интернет 
от друг доставчик, стига да има активен абонамент и стабилна интернет връзка. Възможните комбинации и технологията 
на свързване зависят от адреса.'
    ),
    Document(
        metadata={'Header 1': 'Често задавани въпроси: Vivacom телевизия', 'Header 2': 'Как се заявява EON TV?'},
        page_content='## Как се заявява EON TV?  \nУслугите EON TV и EON SAT могат да се заявят през сайта на 
Vivacom, в магазин или чрез обслужване на клиенти. За предплатената услуга се избира план и брой Smart TV 
приложения онлайн, плаща се с карта при поръчка и след активацията има 30-дневен период за използване.'
    ),
    Document(
        metadata={
            'Header 1': 'Често задавани въпроси: Vivacom телевизия',
            'Header 2': 'На колко устройства може да се гледа?'
        },
        page_content='## На колко устройства може да се гледа?  \nСпоред FAQ страницата лимитите са:  \n- EON TV: 
до 5 телевизионни устройства;\n- EON SAT: до 4 телевизионни устройства;\n- предплатена EON TV: до 2 Smart TV 
приложения;\n- EON TV с интернет от друг доставчик: до 2 телевизионни устройства;\n- мобилното приложение може да 
се използва на до 3 избрани мобилни/компютърни устройства.  \nЗа Smart TV приложението трябва да се провери 
списъкът със съвместими модели.'
    ),
    Document(
        metadata={
            'Header 1': 'Често задавани въпроси: Vivacom телевизия',
            'Header 2': 'Какви функции предлага EON TV?'
        },
        page_content='## Какви функции предлага EON TV?  \nСред описаните възможности са връщане на съдържание до 7
дни назад за избрани канали, видеотека, персонални профили, детски профил и избор между светъл и тъмен интерфейс. 
Наличността на конкретно съдържание и функции зависи от плана и устройството.'
    ),
    Document(
        metadata={
            'Header 1': 'Често задавани въпроси: Vivacom телевизия',
            'Header 2': 'Как се управлява абонаментът?'
        },
        page_content='## Как се управлява абонаментът?  \nЗа промяна на EON TV пакет, добавян

In [ ]:
chroma = Chroma(collection_name="faq", persist_directory="./chroma_db")

In [97]:
chroma.add_documents(chunked_documents)

C:\Users\marij\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [01:52<00:00, 741kiB/s]   


['2f861434-035e-4059-9d0f-eec9445be960',
 'c947e23f-1860-46ee-aeb3-5e895a4b3f1d',
 '26299c28-7d01-4bef-b103-9bc087f865ac',
 'bda15ba9-1ba9-4dec-a313-76bdd230262a',
 'cffef504-18ab-40f9-a7e7-bb79cb812b4f',
 '50ab50d3-fb44-43e6-b501-19cfebe98539',
 'd037f0b2-1ef0-4edd-b807-b1ef5c168f38',
 '332ad62c-8406-498a-9879-c1a3f0cc7b5f',
 '45dd674a-9c8f-4d57-a577-5ed996d7aa92',
 'ec0756a0-b2e0-4266-8416-8ee554b48d77',
 'b484e73f-923a-4bf0-ba65-3e7f31456164',
 '9a39a917-450b-4a1e-85a7-1cf9c187712b',
 'd61f42a9-5cbd-4c69-b5d8-1abb63444dda',
 '0ab463cc-8a81-40fb-88ea-ed351ee49ae0']

In [100]:
chroma.search("Какво е EON?",search_type = "similarity",  k=3)

[Document(id='c947e23f-1860-46ee-aeb3-5e895a4b3f1d', metadata={'Header 2': 'Какво е EON TV?', 'Header 1': 'Често задавани въпроси: Vivacom телевизия'}, page_content='## Какво е EON TV?  \nEON е телевизионна платформа на Vivacom за гледане на живи канали и съдържание по заявка на телевизор, мобилно устройство или компютър. Посочени са пакети EON LIGHT, EON FULL и EON PREMIUM. Услугата може да се предлага като интерактивна телевизия, сателитна телевизия (EON SAT) или предплатена Smart TV услуга.'),
 Document(id='26299c28-7d01-4bef-b103-9bc087f865ac', metadata={'Header 1': 'Често задавани въпроси: Vivacom телевизия', 'Header 2': 'Каква е разликата между EON TV, EON SAT и предплатена EON TV?'}, page_content='## Каква е разликата между EON TV, EON SAT и предплатена EON TV?  \n- **EON TV** е интерактивна телевизия, която може да се комбинира с оптичен интернет от Vivacom или да се използва самостоятелно.\n- **EON SAT** използва сателитен сигнал и може да предлага интерактивни функции при инт

### Create a tool to search the VB

In [ ]:
from langchain.tools import tool

@tool
def search_faq(query: str) -> str:
    """Searches the FAQ document for relevant information."""
    results = chroma.search(query, search_type="similarity", k=3)
    return "\n\n".join([result.page_content for result in results]) 


In [105]:
search_faq.invoke("Какво е EON?")

'## Какво е EON TV?  \nEON е телевизионна платформа на Vivacom за гледане на живи канали и съдържание по заявка на телевизор, мобилно устройство или компютър. Посочени са пакети EON LIGHT, EON FULL и EON PREMIUM. Услугата може да се предлага като интерактивна телевизия, сателитна телевизия (EON SAT) или предплатена Smart TV услуга.\n\n## Каква е разликата между EON TV, EON SAT и предплатена EON TV?  \n- **EON TV** е интерактивна телевизия, която може да се комбинира с оптичен интернет от Vivacom или да се използва самостоятелно.\n- **EON SAT** използва сателитен сигнал и може да предлага интерактивни функции при интернет връзка.\n- **Предплатена EON TV** е безсрочна услуга без дългосрочен договор и без предоставено оборудване; използва се чрез Smart TV приложение.\n\n## Къде да потърся актуална помощ?  \n- [Vivacom: FAQ за телевизия](https://www.vivacom.bg/pomosht/chesto-zadavani-vuprosi/televiziya)\n- [Помощ за EON](https://www.vivacom.bg/eon/pomosht)\n- [My Vivacom](https://my.vivaco

In [ ]:
system_prompt = "You are a helpful assistant that can answer questions and perform tasks."

rag_model =  openai_model.bind_tools(
    [search_faq],
    tool_choice="required",  # or "any", "auto"
)

rag_agent = create_agent(
    model=rag_model,
    tools=[search_faq],
    system_prompt="Use the FAQ search tool before answering.",
    debug=True,
)

In [119]:
messages = [
HumanMessage(content="Какво е EON?'")
]

response = rag_agent.invoke(input ={"messages": messages})
print_conversation(response['messages'])

[values] {'messages': [HumanMessage(content="Какво е EON?'", additional_kwargs={}, response_metadata={}, id='606ad330-be3d-4bae-8e24-d980c374ceb1')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 144, 'total_tokens': 171, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-ELSEOzz2ujp2PxCnMZYoR49rxREyS', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a07bb9-2c27-72d1-94c4-d60e50a78f71-0', tool_calls=[{'name': 'search_faq', 'args': {'query': 'What is EON'}, 'id': 'call_mw

### Transform the VB into a Retriever

In [122]:
from langchain_core.tools import create_retriever_tool

In [123]:
chroma_retriever = chroma.as_retriever(search_type="similarity", search_kwargs={"k": 3})
search_faq_tool = create_retriever_tool(chroma_retriever, name="search_faq", description="Searches the FAQ document for relevant information.")

In [124]:
search_faq_tool.invoke("Какво е EON?")

'## Какво е EON TV?  \nEON е телевизионна платформа на Vivacom за гледане на живи канали и съдържание по заявка на телевизор, мобилно устройство или компютър. Посочени са пакети EON LIGHT, EON FULL и EON PREMIUM. Услугата може да се предлага като интерактивна телевизия, сателитна телевизия (EON SAT) или предплатена Smart TV услуга.\n\n## Каква е разликата между EON TV, EON SAT и предплатена EON TV?  \n- **EON TV** е интерактивна телевизия, която може да се комбинира с оптичен интернет от Vivacom или да се използва самостоятелно.\n- **EON SAT** използва сателитен сигнал и може да предлага интерактивни функции при интернет връзка.\n- **Предплатена EON TV** е безсрочна услуга без дългосрочен договор и без предоставено оборудване; използва се чрез Smart TV приложение.\n\n## Къде да потърся актуална помощ?  \n- [Vivacom: FAQ за телевизия](https://www.vivacom.bg/pomosht/chesto-zadavani-vuprosi/televiziya)\n- [Помощ за EON](https://www.vivacom.bg/eon/pomosht)\n- [My Vivacom](https://my.vivaco

In [125]:
system_prompt = "You are a helpful assistant that can answer questions and perform tasks."

rag_model =  openai_model.bind_tools(
    [search_faq_tool],
    tool_choice="required",  # or "any", "auto"
)

rag_agent = create_agent(
    model=rag_model,
    tools=[search_faq_tool],
    system_prompt="Use the FAQ search tool before answering.",
    debug=True,
)

In [126]:
messages = [
HumanMessage(content="Какво е EON?'")
]

response = rag_agent.invoke(input ={"messages": messages})
print_conversation(response['messages'])

[values] {'messages': [HumanMessage(content="Какво е EON?'", additional_kwargs={}, response_metadata={}, id='0bb5c7d2-b3a8-4862-96be-9b5dfebfb9a2')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 153, 'total_tokens': 178, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-ELSSjpB7mYqUgLB6OdqM8lEy8zjgc', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a07bc6-8949-7371-944c-4c17179e24ad-0', tool_calls=[{'name': 'search_faq', 'args': {'query': 'EON'}, 'id': 'call_RhJFyXUS1X